## Install Required Dependencies

In [2]:
:dep solana-program = { version = "^2.0.2", default_features = false }

# Chapter 6: Writing Solana Programs with Rust and Anchor Framework

### Introduction

Welcome to the sixth chapter of our series on developing Web3 applications using Rust, focusing on the Solana blockchain. In this chapter, we dive deep into writing Solana programs using the Anchor framework. Anchor simplifies Solana program development by abstracting complex operations such as serialization and deserialization instructions, allowing for faster and more efficient development.

### 1. Anchor vs Native Solana Programs

[**The Anchor framework**](https://www.anchor-lang.com/) significantly simplifies Solana program development compared to native approaches. It abstracts away low-level implementation details like instruction serialization and deserialization, enabling us to focus more on application logic rather than blockchain specifics.

#### 1.1 Instruction Structure and Data Handling

In Solana, each transaction contains instructions that specify actions to be performed on the blockchain. These instructions consist of a program ID, a list of accounts involved, and serialized data. Understanding how to pack and unpack instruction data is important for interpreting instructions correctly within Solana programs.

```rust
enum MyInstruction {
    Initialize { data: u32 },
    Transfer { amount: u64 },
}

impl MyInstruction {
    pub fn pack(&self) -> Vec<u8> {
        match self {
            MyInstruction::Initialize { data } => {
                let mut buf = vec![0; 8];
                buf[0] = 0; // Instruction index for Initialize
                buf[1..].copy_from_slice(&data.to_le_bytes());
                buf
            },
            MyInstruction::Transfer { amount } => {
                let mut buf = vec![0; 12];
                buf[0] = 1; // Instruction index for Transfer
                buf[1..].copy_from_slice(&amount.to_le_bytes());
                buf
            },
        }
    }

    pub fn unpack(data: &[u8]) -> Result<Self, ProgramError> {
        match data[0] {
            0 => {
                let num = u32::from_le_bytes(data[1..5].try_into().unwrap());
                Ok(MyInstruction::Initialize { data: num })
            },
            1 => {
                let num = u64::from_le_bytes(data[1..9].try_into().unwrap());
                Ok(MyInstruction::Transfer { amount: num })
            },
            _ => Err(ProgramError::InvalidInstructionData),
        }
    }
}
```

Solana's transaction model relies heavily on instructions, which are encoded into byte arrays and processed by Solana programs. The `MyInstruction` enum demonstrates how different types of instructions can be defined, packed into byte arrays, and unpacked within Solana programs. Packing involves converting instruction data into a byte format suitable for transmission over the blockchain, while unpacking decodes received byte data back into structured instruction types. This process ensures that Solana programs can accurately interpret and execute instructions sent by clients or other programs on the blockchain.

```sh
+-------------------------+
|   Solana Instruction    |
+-------------------------+
| Program ID              |
| Accounts (Pubkeys)      |
| Instruction Data (u8[]) |
+-------------------------+

Instruction Data Layout:
+-----------------+-------------------------------+
| Instruction Tag |       Instruction Data        |
+-----------------+-------------------------------+
|      1 byte     |           4 bytes             |
+-----------------+-------------------------------+

Different Instructions:

+----------------+-----------------+------------------+-------------------------------+
| Instruction    | Instruction Tag |      Purpose     |       Instruction Data        |
+----------------+-----------------+------------------+-------------------------------+
| Initialize     |       0x00      | Initialization   |      4 bytes of data          |
| Transfer       |       0x01      | Transfer Amount  |      8 bytes of amount        |
+----------------+-----------------+------------------+-------------------------------+

Layout for 'Initialize' Instruction:

+-----------+-----------------------------+
| Byte 0    |         0x00 (Tag)          |
+-----------+-----------------------------+
| Byte 1-4  |  Initialization Data (u32)  |
+-----------+-----------------------------+

Layout for 'Transfer' Instruction:

+-----------+-----------------------------+
| Byte 0    |         0x01 (Tag)          |
+-----------+-----------------------------+
| Byte 1-8  |      Transfer Amount (u64)  |
+-----------+-----------------------------+
```

The above illustration demonstrates the structure of a Solana instruction, emphasizing the components relevant to instruction handling within Solana programs. Each instruction includes a program ID that identifies the target program on the blockchain, a list of accounts (represented as public keys) involved in the instruction, and serialized instruction data encoded as a byte array (`u8[]`). Solana programs use this information to determine the action to perform and manipulate relevant blockchain state.

In [17]:
use solana_program::{
    account_info::{next_account_info, AccountInfo},
    entrypoint::ProgramResult,
    msg,
    program_error::ProgramError,
    pubkey::Pubkey,
};

use std::rc::Rc;
use std::cell::RefCell;

// Define the MyInstruction enum
pub enum MyInstruction {
    Initialize { data: u32 },
    Transfer { amount: u64 },
}

impl MyInstruction {
    pub fn pack(&self) -> Vec<u8> {
        match self {
            MyInstruction::Initialize { data } => {
                let mut buf = vec![0; 5];
                buf[0] = 0; // Instruction index for `Initialize`
                buf[1..].copy_from_slice(&data.to_le_bytes());
                buf
            },
            MyInstruction::Transfer { amount } => {
                let mut buf = vec![0; 9];
                buf[0] = 1; // Instruction index for `Transfer`
                buf[1..].copy_from_slice(&amount.to_le_bytes());
                buf
            },
        }
    }

    pub fn unpack(input: &[u8]) -> Result<Self, ProgramError> {
        let (&tag, rest) = input.split_first().ok_or(ProgramError::InvalidInstructionData)?;
        Ok(match tag {
            0 => {
                let (data, _rest) = rest.split_at(4);
                MyInstruction::Initialize {
                    data: u32::from_le_bytes(data.try_into().unwrap()),
                }
            },
            1 => {
                let (amount, _rest) = rest.split_at(8);
                MyInstruction::Transfer {
                    amount: u64::from_le_bytes(amount.try_into().unwrap()),
                }
            },
            _ => return Err(ProgramError::InvalidInstructionData),
        })
    }
}

fn process_instruction(
    program_id: &Pubkey,
    accounts: &[AccountInfo],
    instruction_data: &[u8],
) -> ProgramResult {
    let instruction = MyInstruction::unpack(instruction_data)?;

    match instruction {
        MyInstruction::Initialize { data } => {
            msg!("Instruction: Initialize");
            process_initialize(accounts, data, program_id)
        },
        MyInstruction::Transfer { amount } => {
            msg!("Instruction: Transfer");
            process_transfer(accounts, amount, program_id)
        },
    }
}

fn process_initialize(
    accounts: &[AccountInfo],
    data: u32,
    program_id: &Pubkey,
) -> ProgramResult {
    let account_info_iter = &mut accounts.iter();
    let account = next_account_info(account_info_iter)?;

    if account.owner != program_id {
        msg!("Account does not have the correct program id");
        return Err(ProgramError::IncorrectProgramId);
    }

    let mut account_data = account.try_borrow_mut_data()?;
    account_data[0..4].copy_from_slice(&data.to_le_bytes());

    msg!("Account initialized with data: {}", data);
    Ok(())
}

fn process_transfer(
    accounts: &[AccountInfo],
    amount: u64,
    program_id: &Pubkey,
) -> ProgramResult {
    let account_info_iter = &mut accounts.iter();
    let source_account = next_account_info(account_info_iter)?;
    let destination_account = next_account_info(account_info_iter)?;

    if source_account.owner != program_id {
        msg!("Source account does not have the correct program id");
        return Err(ProgramError::IncorrectProgramId);
    }

    if destination_account.owner != program_id {
        msg!("Destination account does not have the correct program id");
        return Err(ProgramError::IncorrectProgramId);
    }

    let mut source_data = source_account.try_borrow_mut_data()?;
    let mut destination_data = destination_account.try_borrow_mut_data()?;

    let source_balance = u64::from_le_bytes(source_data[0..8].try_into().unwrap());
    let destination_balance = u64::from_le_bytes(destination_data[0..8].try_into().unwrap());

    if source_balance < amount {
        msg!("Insufficient funds in source account");
        return Err(ProgramError::InsufficientFunds);
    }

    source_data[0..8].copy_from_slice(&(source_balance - amount).to_le_bytes());
    destination_data[0..8].copy_from_slice(&(destination_balance + amount).to_le_bytes());

    msg!("Transferred {} from source to destination", amount);
    Ok(())
}

fn main() -> ProgramResult {
    let program_id = Pubkey::new_unique();
    let account1_pubkey = Pubkey::new_unique();
    let account2_pubkey = Pubkey::new_unique();
    let mut lamports1 = 0u64;
    let mut lamports2 = 0u64;

    let mut account1_data = vec![0u8; 8];
    let mut account2_data = vec![0u8; 8];

    let account1_info = AccountInfo {
        key: &account1_pubkey,
        lamports: Rc::new(RefCell::new(&mut lamports1)),
        data: Rc::new(RefCell::new(&mut account1_data[..])),
        owner: &program_id,
        rent_epoch: 0,
        is_signer: false,
        is_writable: true,
        executable: false,
    };

    let account2_info = AccountInfo {
        key: &account2_pubkey,
        lamports: Rc::new(RefCell::new(&mut lamports2)),
        data: Rc::new(RefCell::new(&mut account2_data[..])),
        owner: &program_id,
        rent_epoch: 0,
        is_signer: false,
        is_writable: true,
        executable: false,
    };

    let initialize_instruction_data = MyInstruction::Initialize { data: 42 }.pack();
    let transfer_instruction_data = MyInstruction::Transfer { amount: 10 }.pack();

    process_instruction(
        &program_id,
        &[account1_info.clone()],
        &initialize_instruction_data,
    )?;

    process_instruction(
        &program_id,
        &[account1_info.clone(), account2_info.clone()],
        &transfer_instruction_data,
    )?;

    println!("Account1 data after Initialize: {:?}", account1_data);
    println!("Account2 data after Transfer: {:?}", account2_data);

    Ok(())
}

main()

Instruction: Initialize
Account initialized with data: 42
Instruction: Transfer
Transferred 10 from source to destination
Account1 data after Initialize: [32, 0, 0, 0, 0, 0, 0, 0]
Account2 data after Transfer: [10, 0, 0, 0, 0, 0, 0, 0]


Ok(())

```sh
  +-------------------+                 +------------------------+
  |      Client       |                 |     Solana Program     |
  +-------------------+                 +------------------------+
  |                   |                 |                        |
  |  Send Transaction |  Transfer(10)   |                        |
  |  with Instructions|---------------->|  Receive Transaction   |
  |  Account1: 42     |  Initialize(42) |                        |
  |  Account1: 10     |                 |  Extract Instructions  |
  |                   |                 |                        |
  |                   |                 |  Process Instructions  |
  |                   |                 |                        |
  |                   |                 |  Update Account Data   |
  |                   |                 |                        |
  |                   |   Update        |  Prepare Updated Data  |
  |  Receive Updated  |<----------------|                        |
  |  Account Data     |                 |  Send Updated Data     |
  |  Account1: 32     |                 |                        |
  |  Account2: 10     |                 |                        |
  +-------------------+                 +------------------------+
```

The `main` function begins by setting up example data structures necessary for simulating interactions with the Solana program. This includes generating unique public keys (`program_id`, `account1_pubkey`, `account2_pubkey`) using the [**`Pubkey::new_unique()`**](https://docs.rs/solana-program/latest/solana_program/pubkey/struct.Pubkey.html#method.new_unique) function provided by the Solana Rust SDK. These keys are essential as they uniquely identify entities within the Solana blockchain ecosystem, such as programs and accounts.

Next, account data (`account1_data` and `account2_data`) is initialized as byte vectors. These vectors simulate the mutable data associated with Solana accounts. In a real-world scenario, account data could represent balances, states, or any custom application-specific data that the Solana program interacts with or modifies.

The [**`AccountInfo`**](https://docs.rs/solana-program/latest/solana_program/account_info/struct.AccountInfo.html) structures (`account1_info` and `account2_info`) are then instantiated using the [**`AccountInfo::new()`**](https://docs.rs/solana-program/latest/solana_program/account_info/struct.AccountInfo.html#method.new) function. These structures encapsulate information about Solana accounts, including their public key, ownership status, associated data, and other metadata necessary for Solana programs to operate. The example accounts are configured with ownership by `program_id` to demonstrate how programs can access and manipulate their associated accounts securely.

After initializing accounts, example instructions (`initialize_instruction_data` and `transfer_instruction_data`) are prepared using the `MyInstruction::pack()` method. These instructions are serialized into byte vectors (`Vec<u8>`) representing operations that the Solana program will execute. In this case, `initialize_instruction_data` demonstrates an initialization operation with a specific data payload (`data: 42`), while `transfer_instruction_data` illustrates a transfer operation with an amount (`amount: 10`).

The `process_instruction` function is then invoked twice, simulating the interaction with the Solana program. This function processes the serialized instruction data (`initialize_instruction_data` and `transfer_instruction_data`), unpacks them using `MyInstruction::unpack()`, and executes corresponding logic based on the instruction type.

Finally, the `main` function prints the resulting state of `account1_data` and `account2_data` after executing the instructions. This demonstrates the expected changes or effects of invoking the Solana program, validating the correctness of operations performed by the program on associated accounts.

---

### 2. The Anchor Framework

Setting up an Anchor project involves configuring the necessary dependencies and project structure. Anchor provides a [**CLI**](https://www.anchor-lang.com/docs/cli) for initializing new projects, handling program deployment, and managing client interactions with the blockchain.

#### 2.1 Initializing a New Anchor Project

To create a new Anchor project, use the CLI command `anchor init <program-name> --template multiple --test-template rust`. This command sets up a new Rust project with the necessary dependencies and configuration files for developing Solana programs, alongside a Rust test crate instead of a JavaScript/Typescript client to test the program.

```sh
$ anchor init <program-name> --template multiple --test-template rust
```

The `anchor init` command initializes a new Rust project configured for Solana program development using the Anchor framework. It creates essential files and directories, including `Anchor.toml` and `Cargo.toml`, which manage project dependencies and settings. By utilizing the `--template multiple` option, this command sets up a project structure designed to handle multiple modules for one Solana program within a single program. The `--test-template rust` option ensures that the project includes a Rust test crate, allowing us to write and run tests for our Solana programs in Rust. This initialization process streamlines the setup, ensuring we can quickly start working on Solana programs without the need for extensive manual configuration.

#### 2.2 Project Structure

An Anchor project typically includes configuration files (`Anchor.toml`, `Cargo.toml`), program code (`lib.rs`), and client code (`src/` directory). Understanding these components is crucial for structuring and managing complex Solana applications effectively.

- **`Anchor.toml`**: This file contains configuration settings specific to the Anchor framework, including project metadata, deployment configurations, and account schema definitions. The `Anchor.toml` file centralizes project-specific settings and helps manage interactions with the Anchor framework, ensuring smooth integration and deployment processes.
  
- **`Cargo.toml`**: As with any Rust project, `Cargo.toml` manages dependencies and project settings. It includes dependencies required for Solana program development and deployment, such as the `solana-program` crate and other utility libraries. This file is crucial for maintaining the project's build configuration and ensuring that all necessary dependencies are correctly included and versioned.
  
- **`programs/program_name/src/lib.rs`**: The main program file (`lib.rs`) contains the Solana program logic, defined using Anchor's macros and decorators. This file integrates with Solana's transaction processing system and manages program state and execution flow. Within `lib.rs`, we define program entry points, state transition functions, and error handling logic, leveraging Anchor's abstractions to simplify Solana program development.
  
- **`programs/program_name/src/`**: This directory houses additional Rust modules that complement the main program logic. These modules can include helper functions, utility libraries, states, and modular components that facilitate code organization and reuse. By structuring the codebase into separate modules, we can maintain cleaner, more maintainable code, improving the scalability and readability of the Solana program.

- **`tests/`**: The `tests/` directory contains Rust-based test cases that validate the functionality of the Solana programs. These tests leverage the Rust testing framework to ensure correctness and reliability, providing a robust environment for writing and executing unit tests, integration tests, and end-to-end tests for Solana programs in Rust.

In [3]:
use std::process::{Command, Output, Stdio};

fn execute_command(command: &str) -> Result<(), std::io::Error> {
    let status = Command::new("bash")
        .arg("-c")
        .arg(command)
        .stderr(Stdio::inherit())
        .status()?;

    if status.success() {
        Ok(())
    } else {
        Err(std::io::Error::from_raw_os_error(status.code().unwrap_or(1)))
    }
}

In [3]:
let command = "anchor init my-program --template multiple --test-template rust";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

    Creating library `tests` package
      Adding `tests` as member of workspace at `/home/mahmoud/Desktop/solana/rust-web3-solana/chapter-6/my-program`
note: see more `Cargo.toml` keys and their definitions at https://doc.rust-lang.org/cargo/reference/manifest.html


yarn install v1.22.19


warning package.json: No license field


info No lockfile found.


warning No license field


...

warning mocha > glob@7.2.0: Glob versions prior to v9 are no longer supported
warning mocha > glob > inflight@1.0.6: This module is not supported, and leaks memory. Do not use it. Check out lru-cache if you want a good and tested way to coalesce async requests by a key value, which is much more comprehensive and powerful.


[2/4] Fetching packages...
[3/4] Linking dependencies...
[4/4] Building fresh packages...
success Saved lockfile.
Done in 18.48s.


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


Initialized empty Git repository in /home/mahmoud/Desktop/solana/rust-web3-solana/chapter-6/my-program/.git/
my-program initialized


()

---

### 2.3. Writing Solana Programs with Anchor

Anchor is designed to simplify the development of Solana programs. It abstracts some of the complexities involved in writing on-chain programs by providing various tools and utilities that make common tasks easier, such as account management, instruction handling, and state initialization.

#### 2.3.1 Defining Program Entry Points

In Anchor, the `#[program]` macro is used to define the module that contains all of the program's instructions.

```rust
use anchor_lang::prelude::*;

#[program]
pub mod my_program {
    use super::*;

    pub fn initialize(ctx: Context<Initialize>) -> Result<()> {
        initialize::handler(ctx)
    }

    pub fn transfer(ctx: Context<Transfer>, amount: u64) -> Result<()> {
        transfer::handler(ctx, amount)
    }
}
```

- **`#[program]` Macro**: This macro specifies the module containing all of the program’s instructions. In the example, `my_program` is the module that includes the `initialize` and `transfer` functions.

- **Entry Points**: Within the `my_program` module, functions like `initialize` and `transfer` serve as entry points for various program instructions:
  - **`initialize`**: Sets up initial state for accounts. It calls the `initialize::handler` function to execute the initialization logic.
  - **`transfer`**: Manages the transfer of funds between accounts. It invokes the `transfer::handler` function to perform the transfer operation.

#### 2.3.2 Handling Program States

State management is crucial for blockchain programs. Anchor simplifies this by allowing you to define stateful structs using the `#[account]` attribute, which handles serialization and deserialization automatically.

```rust
use anchor_lang::prelude::*;

#[account]
#[derive(Default)]
/// The main account structure for the program, holding authority and data.
pub struct MyAccount {
    /// The public key of the authority (owner) of the account.
    pub authority: Pubkey,
    /// The data associated with the account.
    pub data: u64,
}
```

- **`#[account]` Attribute**: Annotates Rust structs that represent Solana accounts. Anchor generates code to handle serialization and deserialization of these structs.

- **State Struct**: `MyAccount` represents the state stored in an account:
  - **`authority`**: Public key of the account’s owner.
  - **`data`**: An arbitrary piece of data, represented as a `u64`.

#### 2.3.3 Defining Instruction Contexts

Instruction contexts describe the accounts involved in a particular instruction and their roles.

```rust
use anchor_lang::prelude::*;

#[derive(Accounts)]
#[instruction()]
/// Context for initializing a new MyAccount.
pub struct Initialize<'info> {
    /// The account of the user initializing the MyAccount.
    #[account(mut)]
    pub authority: Signer<'info>,

    /// The MyAccount to be initialized.
    #[account(
        init,
        payer = authority,
        space = 8 + std::mem::size_of::<MyAccount>(),
    )]
    pub my_account: Box<Account<'info, MyAccount>>,

    /// The system program, required for account creation.
    pub system_program: Program<'info, System>,

    /// The rent sysvar, which holds the rent exemption information.
    pub rent: Sysvar<'info, Rent>,
}

#[derive(Accounts)]
#[instruction()]
/// Context for transferring data between two MyAccounts.
pub struct Transfer<'info> {
    /// The MyAccount to transfer data from.
    #[account(
        mut,
        has_one = authority,
    )]
    pub from: Box<Account<'info, MyAccount>>,

    /// The MyAccount to transfer data to.
    #[account(mut)]
    pub to: Box<Account<'info, MyAccount>>,

    /// The account of the user performing the transfer.
    #[account(mut)]
    pub authority: Signer<'info>,
}
```

- **`#[derive(Accounts)]` Macro**: Used to define contexts for instructions, detailing which accounts are involved and their specific roles.

- **Contexts**:
  - **`Initialize`**: Sets up the initial account:
    - **`authority`**: The signer of the transaction.
    - **`my_account`**: The account being initialized.
    - **`system_program`** and **`rent`**: Required for account creation and rent exemption.
  - **`Transfer`**: Handles the data transfer:
    - **`from`** and **`to`**: Accounts participating in the transfer.
    - **`authority`**: The signer authorizing the transfer.

#### 2.3.4 Handling Instructions

Instructions are the operations that a Solana program executes. Anchor automates the serialization and deserialization of instruction data.

```rust
use anchor_lang::prelude::*;
use crate::state::Transfer;
use crate::error::ErrorMsg;

pub fn handler(ctx: Context<Transfer>, amount: u64) -> Result<()> {
    let from_account = &mut ctx.accounts.from;
    let to_account = &mut ctx.accounts.to;

    if from_account.data < amount {
        return Err(ErrorMsg::InsufficientFunds.into());
    }

    from_account.data -= amount;
    to_account.data += amount;

    Ok(())
}
```

- **Instruction Handler**: This function processes the instructions provided to the program:
  - **Context Parameter**: `Context<Transfer>` contains accounts involved in the instruction.
  - **Instruction Logic**: Validates that `from_account` has sufficient funds. If not, it returns an `InsufficientFunds` error. Otherwise, it updates account balances.

#### 2.3.5 Error Handling and Program Execution

Error handling in Anchor leverages Rust's error handling features to manage and report errors effectively.

```rust
use anchor_lang::prelude::*;

#[error_code]
pub enum ErrorMsg {
    #[msg("Insufficient Funds")]
    InsufficientFunds,
}
```

- **Custom Error Types**: Errors are defined using `#[error_code]` and `enum`. Each variant, like `InsufficientFunds`, represents a specific error condition.

- **Error Handling**: The `handler` function uses these error types to provide clear error messages and handle issues gracefully.

#### 2.3.6 Client Interactions with Anchor

Anchor provides tools to build client applications that interact with Solana programs. Clients send transactions and query blockchain states using generated bindings.

```sh
$ anchor test
```

- **Integration Testing**: The `anchor test` command allows developers to run tests for client interactions with Solana programs to ensure client-side logic and program execution are functioning correctly.

In [4]:
// Build, deploy program to localnet and run integration tests against a local validor
let command = "cd my-program && anchor test";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

    Finished release [optimized] target(s) in 0.34s
    Finished `test` profile [unoptimized + debuginfo] target(s) in 0.39s
     Running unittests src/lib.rs (/home/mahmoud/Desktop/solana/rust-web3-solana/chapter-6/my-program/target/debug/deps/my_program-6ca604ef98986aeb)



Found a 'test' script in the Anchor.toml. Running it as a test suite!

Running test suite: "/home/mahmoud/Desktop/solana/rust-web3-solana/chapter-6/my-program/Anchor.toml"



  --> tests/src/test_ix.rs:10:5
   |
10 |     instruction,
   |     ^^^^^^^^^^^
   |
   = note: `#[warn(unused_imports)]` on by default

    Finished `test` profile [unoptimized + debuginfo] target(s) in 0.53s
     Running unittests src/lib.rs (target/debug/deps/my_program-f73c45ae1e479cef)



running 1 test
test test_id ... ok

test result: ok. 1 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s



     Running unittests src/lib.rs (target/debug/deps/tests-45b502e5c7fe059e)


...

   Doc-tests my_program



running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s



   Doc-tests tests



running 0 tests

test result: ok. 0 passed; 0 failed; 0 ignored; 0 measured; 0 filtered out; finished in 0.00s



()

#### 2.3.7 Smart Contract Deployment

Deploying a smart contract on Solana involves compiling and uploading the program to the blockchain.

```sh
$ anchor build
$ anchor deploy
```

- **Deployment Commands**:
  - **`anchor build`**: Compiles the smart contract into a binary format compatible with Solana.
  - **`anchor deploy`**: Uploads the compiled binary to the Solana blockchain and sets up the program’s initial state.


**In a separate terminal, execute `solana-test-validator` to run a local validator on your machine.**

In [22]:
// Initialize a new wallet
let command = "solana config set --url localhost && solana-keygen new --force";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

Config File: /home/mahmoud/.config/solana/cli/config.yml
RPC URL: http://localhost:8899 
WebSocket URL: ws://localhost:8900/ (computed)
Keypair Path: /home/mahmoud/.config/solana/mainnet.json 
Commitment: confirmed 
Generating a new keypair


thread 'main' panicked at keygen/src/keygen.rs:491:92:
called `Result::unwrap()` on an `Err` value: Os { code: 6, kind: Uncategorized, message: "No such device or address" }
stack backtrace:
   0: rust_begin_unwind
             at /rustc/82e1608dfa6e0b5569232559e3d385fea5a93112/library/std/src/panicking.rs:645:5
   1: core::panicking::panic_fmt
             at /rustc/82e1608dfa6e0b5569232559e3d385fea5a93112/library/core/src/panicking.rs:72:14
   2: core::result::unwrap_failed
             at /rustc/82e1608dfa6e0b5569232559e3d385fea5a93112/library/core/src/result.rs:1653:5
   3: solana_keygen::do_main
   4: solana_keygen::main
note: Some details are omitted, run with `RUST_BACKTRACE=full` for a verbose backtrace.
Error executing command: Network is unreachable (os error 101)


()

In [26]:
// Get pubkey
let command = "solana address -k /home/mahmoud/.config/solana/id.json";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

3f76fxPY9vWJCZVVc2fRsdVxrakWHGUe2LaQ1DZnCBxK


()

In [24]:
// Airdrop some SOL into the generated account
let command = "solana airdrop 2 3f76fxPY9vWJCZVVc2fRsdVxrakWHGUe2LaQ1DZnCBxK --url localhost";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}


Requesting airdrop of 2 SOL

Signature: 4rR1GFK9pQdRguGCRcHf8y8TYnjqYhfVyrebCt51xZvTGQEEVbTbLe5SFVFDVcPecjcs3FEbnPWhR4HJhtUXR4Qb

4.53282296 SOL


()

In [25]:
let command = "cd my-program && solana config set --url localhost && anchor deploy --provider.cluster localnet";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

Config File: /home/mahmoud/.config/solana/cli/config.yml
RPC URL: http://localhost:8899 
WebSocket URL: ws://localhost:8900/ (computed)
Keypair Path: /home/mahmoud/.config/solana/mainnet.json 
Commitment: confirmed 
Deploying cluster: http://127.0.0.1:8899
Upgrade authority: /home/mahmoud/.config/solana/id.json
Deploying program "my_program"...
Program path: /home/mahmoud/Desktop/solana/rust-web3-solana/chapter-6/my-program/target/deploy/my_program.so...
Program Id: Edz73jF4fNGwHoz5Qmf8V5bExfoYgYYBwmNt8kqdUw69

Deploy success


()

#### 2.3.8 Interacting with the Program Using IDL

Anchor's Interface Definition Language (IDL) helps in interacting with deployed programs by providing a JSON representation of the program's API.

```sh
$ anchor idl fetch <ADDRESS> -o my_program.json
```

- **Fetching IDL**: The `anchor idl fetch` command retrieves the IDL for a program and saves it to a file, which can be used for client-side interactions.

In [ ]:
let command = "cd my-program && anchor idl fetch --provider.cluster localnet -o my_program.json Edz73jF4fNGwHoz5Qmf8V5bExfoYgYYBwmNt8kqdUw69";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

In [33]:
let command = "cd my-program && cat target/idl/my_program.json";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

{
  "address": "Edz73jF4fNGwHoz5Qmf8V5bExfoYgYYBwmNt8kqdUw69",
  "metadata": {
    "name": "my_program",
    "version": "0.0.1",
    "spec": "0.1.0",
    "description": "Created with Anchor"
  },
  "instructions": [
    {
      "name": "initialize",
      "discriminator": [
        175,
        175,
        109,
        31,
        13,
        152,
        155,
        237
      ],
      "accounts": [
        {
          "name": "authority",
          "docs": [
            "The account of the user initializing the MyAccount."
          ],
          "writable": true,
          "signer": true
        },
        {
          "name": "my_account",
          "docs": [
            "The MyAccount to be initialized."
          ],
          "writable": true,
          "signer": true
        },
        {
          "name": "system_program",
          "docs": [
            "The system program, which is required for account operations."
          ],
          "address": "1111111111111111111111111111

()

        52,
        200,
        231,
        140,
        3,
        69,
        186
      ],
      "accounts": [
        {
          "name": "from",
          "docs": [
            "The MyAccount to transfer data from."
          ],
          "writable": true
        },
        {
          "name": "to",
          "docs": [
            "The MyAccount to transfer data to."
          ],
          "writable": true
        },
        {
          "name": "authority",
          "docs": [
            "The account of the user performing the transfer."
          ],
          "writable": true,
          "signer": true,
          "relations": [
            "from",
            "to"
          ]
        },
        {
          "name": "system_program",
          "docs": [
            "The system program, which is required for account operations."
          ],
          "address": "11111111111111111111111111111111"
        }
      ],
      "args": [
        {
          "name": "amount",
          "ty

In Rust, you can use the IDL to interact with the program using `anchor_client`:

```rust
use anchor_client::{
    solana_sdk::{
        commitment_config::CommitmentConfig,
        signature::{read_keypair_file, Keypair, Signer},
    },
    Client, Cluster, Program,
};
use my_program::{accounts::{Initialize, Transfer}, instruction};
use std::sync::Arc;

fn setup_program() -> (Client<Arc<Keypair>>, Program<Arc<Keypair>>, Keypair) {
    let anchor_wallet = std::env::var("ANCHOR_WALLET").unwrap();
    let payer = Arc::new(read_keypair_file(&anchor_wallet).unwrap());
    let client = Client::new_with_options(
        Cluster::Localnet,
        Arc::clone(&payer),
        CommitmentConfig::confirmed(),
    );
    let program = client.program(my_program::id()).unwrap();

    (client, program, payer.insecure_clone())
}

fn transfer_account() {
    let (_client, program, authority) = setup_program();

    let from_account = Keypair::new();
    let to_account = Keypair::new();

    // Initialize 'from' account
    let _tx_init_from = program
        .request()
        .accounts(Initialize {
            authority: authority.pubkey(),
            my_account: from_account.pubkey(),
            system_program: my_program::id(),
        })
        .signer(&authority)
        .send()
        .expect("Failed to send initialize 'from' account transaction");

    // Initialize 'to' account
    let _tx_init_to = program
        .request()
        .accounts(Initialize {
            authority: authority.pubkey(),
            my_account: to_account.pubkey(),
            system_program: my_program::id(),
        })
        .signer(&authority)
        .send()
        .expect("Failed to send initialize 'to' account transaction");

    // Perform transfer
    let tx = program
        .request()
        .accounts(Transfer {
            authority: authority.pubkey(),
            from: from_account.pubkey(),
            to: to_account.pubkey(),
            system_program: my_program::id(),
        })
        .signer(&authority)
        .send()
        .expect("Failed to send transfer transaction");

    println!("Transfer transaction signature: {}", tx);
}
```

- **Client Code**: Demonstrates how to interact with the program using IDL-generated bindings:
  - **Setup**: Configures the `Client` and `Program` for interaction.
  - **Transaction Execution**: Initializes accounts and performs a transfer operation.

#### 2.3.9 Smart Contract Upgrades

Upgrading a smart contract involves deploying a new version while managing state migrations.

```sh
$ anchor upgrade --program-id <PROGRAM_ID> <PROGRAM_FILEPATH>
```

- **Upgrade Command**: Deploys a new version of the smart contract and handles state migrations as necessary.

In [32]:
let command = "cd my-program && anchor upgrade --program-id Edz73jF4fNGwHoz5Qmf8V5bExfoYgYYBwmNt8kqdUw69 target/deploy/my_program.so";

if let Err(err) = execute_command(command) {
    eprintln!("Error executing command: {}", err);
}

Program Id: Edz73jF4fNGwHoz5Qmf8V5bExfoYgYYBwmNt8kqdUw69



()

---
### Conclusion

In this chapter, we explored the fundamentals of writing Solana programs using the Anchor framework. We discussed the advantages of using Anchor over native Solana programming methods, explored program structure, instruction handling, state management, error handling, client interactions, and the lifecycle of smart contracts on the Solana blockchain. By leveraging Anchor's powerful abstractions and tools, we can streamline Solana program development, enhance program reliability, and accelerate dApp deployment on the Solana blockchain.

---
---